In [ ]:
!pip install docling-core[chunking]
!pip install docling-core[chunking-openai]
!pip install docling
!pip install -qU pip docling transformers

In [1]:
import logging
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TableFormerMode,
    PdfPipelineOptions
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.layout_model_specs import DOCLING_LAYOUT_HERON, DOCLING_LAYOUT_HERON_101, DOCLING_LAYOUT_EGRET_MEDIUM, DOCLING_LAYOUT_V2, LAYOUTLMV3_BASE
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

# Configure accelerator options for GPU


_log = logging.getLogger(__name__)

# Faster rendering
IMAGE_RESOLUTION_SCALE = 2.0

input_doc_path = r"C:\Users\hp\OneDrive\Desktop\rag_bot\middle_10_pages.pdf"


def main():
    logging.basicConfig(level=logging.INFO)

    # Use threaded options (important)
    pipeline_options = PdfPipelineOptions()

    # ---------- OCR ----------
    pipeline_options.do_ocr = False  

    # ---------- Layout ----------
    pipeline_options.layout_options.model_spec = DOCLING_LAYOUT_HERON

    pipeline_options.do_table_structure = False
    pipeline_options.table_structure_options.do_cell_matching = False
    #pipeline_options.table_structure_options.mode = TableFormerMode.FAST

    # ---------- Images (disable for speed) ----------
    #pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
    pipeline_options.generate_page_images = False
    pipeline_options.generate_picture_images = False
    # ---------- Disable enrichment (prevents heavy backend loading) ----------
    pipeline_options.do_formula_enrichment = False
    pipeline_options.do_code_enrichment = False
    pipeline_options.do_picture_description = False
    pipeline_options.do_picture_classification = False

    # ---------- Memory optimization ----------
    pipeline_options.generate_parsed_pages = False

    # ---------- Thread tuning (optional speed boost) ----------

    # Create converter
    doc_converter = DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    return doc_converter

c:\Users\hp\OneDrive\Desktop\rag_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
doc_converter = main()

In [ ]:
#Load all the necessary models and initialize the pipeline (important to do this before processing documents to avoid loading models during processing which can cause timeouts)
import time
start_time = time.time()
doc_converter.initialize_pipeline(InputFormat.PDF)
init_runtime = time.time() - start_time
_log.info(f"Pipeline initialized in {init_runtime:.2f} seconds.")

In [ ]:
# Process document
start_time = time.time()
conv_res = doc_converter.convert(input_doc_path)

In [51]:
import pickle
import copy
from hierarchical.postprocessor import ResultPostprocessor

with open("pkl_files/conv_res_middle_10_pages.pkl", "rb") as f:  # rb = read binary
    conv_res_middle_10_pages = pickle.load(f)

In [3]:
import pickle
import copy

with open("pkl_files/conv_res_Reference_1_after_post_processing.pkl", "rb") as f:  # rb = read binary
    conv_res_Reference_1 = pickle.load(f)


In [ ]:
import pickle
import copy
with open("conv_res_judgeFlow.pkl", "rb") as f:  # rb = read binary
    conv_res_judgeFlow = pickle.load(f)


In [29]:
import pickle
import copy

with open("pkl_files/conv_res_Boericke_materia_medica.pkl", "rb") as f:  # rb = read binary
    conv_res_Boericke_materia_medica = pickle.load(f)

In [ ]:
import json
print(json.dumps(list(conv_res_Boericke_materia_medica.document.__dict__.keys()), indent=2))

In [ ]:
section_headers = []

for item, level in conv_res_middle_10_pages.document.iterate_items():
    label = getattr(item, "label", None)
    text = getattr(item, "text", "").strip()
    
    if not text:
        continue
        
    if label == "section_header":
        h_level = getattr(item, "level")
        header_entry = {
            "level": h_level,
            "text": text,
            "markdown": f"{'#' * h_level} {text}"
        }
        section_headers.append(header_entry)

# Print all section headers
print("Section Headers found:")
for header in section_headers:
    print(f"Level {header['level']}: {header['text']}")

In [ ]:
print(dict['texts'])

14935


In [84]:
import pickle
import copy

with open("pkl_files/conv_res_Boericke_materia_medica_after_post_processing.pkl", "rb") as f:  # rb = read binary
    conv_res_Boericke_materia_medica_after_post_processing = pickle.load(f)


In [6]:
import logging
from collections import Counter
from docling_core.types.doc import DocItemLabel, SectionHeaderItem

log = logging.getLogger(__name__)

def find_repeating_headers(doc, min_repeat: int = 4) -> set:
    """
    Collect all SECTION_HEADER texts that appear multiple times
    — these are likely running page headers.
    """
    header_texts = []
    for item, _ in doc.iterate_items():
        if item.label == DocItemLabel.SECTION_HEADER:
            header_texts.append(item.text.strip())

    counts = Counter(header_texts)
    return {text for text, count in counts.items() if count >= min_repeat}

def is_false_page_header(item, doc, repeating_headers: set, threshold: float = 0.08) -> bool:
    """Check if an item is a false page header (repeated text or at top of page)."""
    if item.text.strip() in repeating_headers:
        return True
    for prov in item.prov:
        page = doc.pages.get(prov.page_no)
        if page is None:
            continue
        if prov.bbox.t >= page.size.height * (1 - threshold):
            return True
    return False

def remove_false_headers_from_doc(doc, repeating_headers: set, threshold: float = 0.08):
    """
    Removes false page headers from the docling document.
    Re-parents their children to the preceding real heading before deleting.
    """

    # ── Step 1: Identify all false headers ──────────────────────────────────
    false_headers = []
    false_header_refs = set()

    for item, _ in doc.iterate_items():
        if item.label == DocItemLabel.SECTION_HEADER:
            if is_false_page_header(item, doc, repeating_headers, threshold):
                false_headers.append(item)
                false_header_refs.add(item.self_ref)
                log.info(f"Flagged for removal → '{item.text.strip()}'")

    if not false_headers:
        log.info("No false headers found to remove.")
        return doc

    # ── Step 2: Re-parent children of each false header ─────────────────────
    for header in false_headers:
        if not header.children:
            continue # no children to re-parent, skip

        # Find the parent node of this false header
        parent_node = header.parent.resolve(doc=doc)

        # Find the index of this false header among its parent's children
        header_ref = header.get_ref()
        header_index = None
        for i, child_ref in enumerate(parent_node.children):
            if child_ref.cref == header_ref.cref:
                header_index = i
                break

        if header_index is None:
            log.warning(f"Could not find header '{header.text.strip()}' in parent's children, skipping re-parent.")
            continue

        # Walk backwards through siblings to find the nearest real heading
        real_heading = None
        for i in range(header_index - 1, -1, -1):
            sibling = parent_node.children[i].resolve(doc)
            if (isinstance(sibling, SectionHeaderItem)
                    and sibling.self_ref not in false_header_refs):
                real_heading = sibling
                break

        if real_heading:
            # Move each child of the false header to the real heading
            for child_ref in list(header.children): # copy list since _move_subtree modifies it
                child = child_ref.resolve(doc)
                doc._move_subtree(old_subroot=child, new_subroot=real_heading)
            log.info(f"Re-parented children of '{header.text.strip()}' → under '{real_heading.text.strip()}'")
        else:
            # No preceding real heading found — shift children up to the parent level
            log.info(f"No preceding heading found for '{header.text.strip()}', shifting children up.")
            doc._shift_up(old_subroot=header)
            # _shift_up already removes the header from its parent's children,
            # so we should NOT delete it again via delete_items.
            # Remove it from our false_headers list to avoid double-removal.
            false_header_refs.discard(header.self_ref)

    # ── Step 3: Delete the now-childless false headers ──────────────────────
    # Only delete headers that weren't already removed by _shift_up
    headers_to_delete = [h for h in false_headers if h.self_ref in false_header_refs]
    if headers_to_delete:
        doc.delete_items(node_items=headers_to_delete)

    log.info(f"Successfully removed {len(false_headers)} false header(s) from document.")
    return doc


In [5]:
from docling_core.types.doc import DocItemLabel
from collections import Counter
def find_repeating_headers(doc, min_repeat: int = 4) -> set:
    """
    Collect all SECTION_HEADER texts that appear on
    multiple pages — these are likely running page headers.
    """
    header_texts = []

    for item, _ in doc.iterate_items():
        if item.label == DocItemLabel.SECTION_HEADER:
            header_texts.append(item.text.strip())

    counts = Counter(header_texts)
    repeating = {text for text, count in counts.items() if count >= min_repeat}


    return repeating

In [9]:
repeating_headers = find_repeating_headers(conv_res_Reference_1, min_repeat=4)

# Step 2: Remove them directly from the doc object
doc = remove_false_headers_from_doc(conv_res_Reference_1, repeating_headers, threshold=0.08)

In [10]:
def validate_docling_document(doc):
    """
    Full cross-reference consistency check for DoclingDocument.
    Run this before passing to HybridChunker.
    """
    errors = []

    # --- Step 1: Build ground truth index maps ---
    # self_ref must match actual position
    for i, item in enumerate(doc.texts):
        expected = f"#/texts/{i}"
        if item.self_ref != expected:
            errors.append(f"doc.texts[{i}].self_ref='{item.self_ref}' expected '{expected}'")

    for i, item in enumerate(doc.tables):
        expected = f"#/tables/{i}"
        if item.self_ref != expected:
            errors.append(f"doc.tables[{i}].self_ref='{item.self_ref}' expected '{expected}'")

    for i, item in enumerate(doc.groups):
        expected = f"#/groups/{i}"
        if item.self_ref != expected:
            errors.append(f"doc.groups[{i}].self_ref='{item.self_ref}' expected '{expected}'")

    # --- Step 2: Build resolution lookup ---
    ref_map = {}
    for i, item in enumerate(doc.texts):
        ref_map[f"#/texts/{i}"] = item
    for i, item in enumerate(doc.tables):
        ref_map[f"#/tables/{i}"] = item
    for i, item in enumerate(doc.pictures):
        ref_map[f"#/pictures/{i}"] = item
    for i, item in enumerate(doc.groups):
        ref_map[f"#/groups/{i}"] = item
    ref_map["#/body"] = doc.body
    ref_map["#/furniture"] = doc.furniture

    def check_ref(ref_str, context):
        if ref_str not in ref_map:
            errors.append(f"Dangling ref '{ref_str}' in {context}")
            return False
        return True

    # --- Step 3: Walk every node's .children and .parent ---
    all_nodes = (
        [(f"#/texts/{i}", item) for i, item in enumerate(doc.texts)]
        + [(f"#/tables/{i}", item) for i, item in enumerate(doc.tables)]
        + [(f"#/pictures/{i}", item) for i, item in enumerate(doc.pictures)]
        + [(f"#/groups/{i}", item) for i, item in enumerate(doc.groups)]
        + [("#/body", doc.body), ("#/furniture", doc.furniture)]
    )

    for own_ref, node in all_nodes:
        # Check parent pointer is resolvable
        if node.parent is not None:
            parent_cref = node.parent.cref
            if check_ref(parent_cref, f"{own_ref}.parent"):
                # Check that parent actually lists this node as child
                parent_node = ref_map[parent_cref]
                child_crefs = [c.cref for c in parent_node.children]
                if own_ref not in child_crefs:
                    errors.append(
                        f"{own_ref}.parent='{parent_cref}' but parent does NOT list {own_ref} in .children"
                    )

        # Check all children are resolvable
        for child_ref in node.children:
            if check_ref(child_ref.cref, f"{own_ref}.children"):
                # Check child's parent points back
                child_node = ref_map[child_ref.cref]
                if child_node.parent is None or child_node.parent.cref != own_ref:
                    errors.append(
                        f"{own_ref}.children has '{child_ref.cref}' but its .parent='{getattr(child_node.parent, 'cref', None)}'"
                    )

    # --- Step 4: Detect items in doc.texts not reachable from body tree ---
    reachable = set()
    def walk(node):
        reachable.add(node.self_ref)
        for child_ref in node.children:
            if child_ref.cref in ref_map:
                walk(ref_map[child_ref.cref])
    walk(doc.body)

    for i, item in enumerate(doc.texts):
        if item.self_ref not in reachable:
            errors.append(f"doc.texts[{i}] (self_ref='{item.self_ref}') is UNREACHABLE from doc.body")

    if errors:
        print(f"❌ Found {len(errors)} consistency error(s):")
        for e in errors:
            print(f"  - {e}")
    else:
        print("✅ DoclingDocument is consistent.")

    return errors

In [12]:
print(validate_docling_document(doc))

C:\Users\hp\AppData\Local\Temp\ipykernel_16692\3101338657.py:36: DeprecationWarning: deprecated
  ref_map["#/furniture"] = doc.furniture
C:\Users\hp\AppData\Local\Temp\ipykernel_16692\3101338657.py:50: DeprecationWarning: deprecated
  + [("#/body", doc.body), ("#/furniture", doc.furniture)]


✅ DoclingDocument is consistent.
[]


In [ ]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

from docling.chunking import HybridChunker, HierarchicalChunker

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 1000  # set to a small number for illustrative purposes

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS,  # optional, by default derived from `tokenizer` for HF case
)

In [14]:
#Document stores data → Serializer decides how to present it
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
    TripletTableSerializer
)
from docling_core.transforms.serializer.markdown import MarkdownParams


class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            params=MarkdownParams(
                image_placeholder="<!-- image -->"
            ),
        )
    
chunker = HybridChunker(
    tokenizer=tokenizer,
    serializer_provider=MDTableSerializerProvider(),
    merge_peers=True,  # optional, defaults to True
)

chunk_iter = chunker.chunk(dl_doc=doc)
chunks = list(chunk_iter)

In [ ]:
for chunk in chunks[0:]:
    print("---- CHUNK START ----")
    print(chunker.contextualize(chunk=chunk))
    print("---- CHUNK END ----\n\n")

In [ ]:
for idx, chunk in enumerate(chunks):
    if hasattr(chunk.meta, "headings") and chunk.meta.headings:
        heading = ','.join(chunk.meta.headings[0:])
        print(f"{heading}")

In [ ]:
from sentence_transformers import SentenceTransformer
import math
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
tokenizer = model.tokenizer

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is en

In [27]:
data_objects = []
threshold = 200

for idx, chunk in enumerate(chunks):
    text = chunk.text.strip()

    # Step 1: Filter small chunks
    if len(text) <= threshold:
        continue

    # Step 3: Extract heading
    heading = None
    if hasattr(chunk.meta, "headings") and chunk.meta.headings:
        heading = chunk.meta.headings[0]

    # Step 4: Extract doc_items info (page_no + reference)
    page_no = None

    if chunk.meta.doc_items:
        first_item = chunk.meta.doc_items[0]

        # page number
        if first_item.prov:
            page_no = first_item.prov[0].page_no
    
    filename = None
    if chunk.meta.origin and chunk.meta.origin.filename:
        filename = chunk.meta.origin.filename

    # Step 5: Build dictionary
    properties = {
        "Heading": heading,
        "Content": text,
        "Page_No": page_no,
        "Filename": filename
    }
    vector = model.encode(text,task="retrieval")  # Convert numpy array to list for JSON serialization
    norm = math.sqrt(sum(x*x for x in vector))
    #cosine similarity is just a extension of dot product, here i am intentionally normalizing the vector to unit length.
    normalized_vector = [x / norm for x in vector]
    data_object = {
        "properties": properties,
        "vector": normalized_vector
    }
    data_objects.append(data_object)

print("Total data objects:", len(data_objects))

KeyboardInterrupt: 

In [ ]:
for i in data_objects:
    print("---------chunk started--------------")
    print("This the heading - ",i["properties"]["Heading"])
    print("This is the content - ",i["properties"]["Content"])
    print("---------chunk ended--------------\n")

In [90]:
import pickle
import copy
with open("pkl_files/conv_res_Boericke_materia_medica_after_post_processing_waveiate_object.pkl", "wb") as f:  # rb = read binary
    pickle.dump(data_objects, f)


In [ ]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization, StopwordsPreset, VectorDistances
from weaviate.util import generate_uuid5

# Step 1.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:
    # Step 1.2: Create a collection
    movies = client.collections.create(
        name="Reference_1_docs",
        vector_config=Configure.Vectors.self_provided(
            name="Content",
            vector_index_config=Configure.VectorIndex.hnsw(
                distance_metric=VectorDistances.DOT,
                ef=-1,                  # Enable dynamic ef
                dynamic_ef_factor=15,   # Multiplier
                dynamic_ef_min=200,     # Minimum threshold
                dynamic_ef_max=1000,
                ef_construction=128,   # Default: 128
                max_connections=32,    # Default: 32
            ),
            quantizer=Configure.VectorIndex.Quantizer.rq(bits=8)
            #dont know exactly how RQ is better than PQ
        ),
        properties=[
        Property(
            name="Heading",
            data_type=DataType.TEXT,
            tokenization=Tokenization.WORD,
            #"Stopwords are only removed when tokenization is set to word."
            index_searchable=True
        ),
        Property(
            name="Content",
            data_type=DataType.TEXT,
        ),
        Property(
            name="Page_No",
            data_type=DataType.INT
        ),
        Property(
            name="Filename",
            data_type=DataType.TEXT,
            tokenization=Tokenization.WORD,
            index_filterable=True
        )
    ],
    inverted_index_config=Configure.inverted_index(
        bm25_b=0.7,
        bm25_k1=1.25,
        index_null_state=True,
        index_property_length=True,
        index_timestamps=True,
        stopwords_preset=StopwordsPreset.EN,
        stopwords_additions=["1","2","3","4","5","6","7","8","9","0"],
    ),
    )

    reference_1_collection = client.collections.use("Reference_1_docs")
    with reference_1_collection.batch.fixed_size(batch_size=200) as batch:
        for obj in data_objects:
            batch.add_object(properties=obj['properties'], vector=obj['vector'], uuid=generate_uuid5(obj['properties']))
            if batch.number_errors > 10:
                print("Batch import stopped due to excessive errors.")
                break 
    failed_objects = reference_1_collection.batch.failed_objects
    if failed_objects:
        print(f"Number of failed imports: {len(failed_objects)}")
        print(f"First failed object: {failed_objects[0]}")
    print(f"Imported & vectorized {len(reference_1_collection)} objects into the Reference_1_docs collection")

Imported & vectorized 1315 objects into the Reference_1_docs collection


In [94]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization, StopwordsPreset, VectorDistances
from weaviate.util import generate_uuid5

# Step 1.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:
    # Step 1.2: Create a collection
    movies = client.collections.create(
        name="Boericke_materia_medica_docs",
        vector_config=Configure.Vectors.self_provided(
            name="Content",
            vector_index_config=Configure.VectorIndex.hnsw(
                distance_metric=VectorDistances.DOT,
                ef=-1,                  # Enable dynamic ef
                dynamic_ef_factor=15,   # Multiplier
                dynamic_ef_min=200,     # Minimum threshold
                dynamic_ef_max=1000,
                ef_construction=128,   # Default: 128
                max_connections=32,    # Default: 32
            )
        ),
        properties=[
        Property(
            name="Heading",
            data_type=DataType.TEXT,
            tokenization=Tokenization.WORD,
            #"Stopwords are only removed when tokenization is set to word."
            index_searchable=True
        ),
        Property(
            name="Content",
            data_type=DataType.TEXT,
        ),
        Property(
            name="Page_No",
            data_type=DataType.INT
        ),
        Property(
            name="Filename",
            data_type=DataType.TEXT,
            tokenization=Tokenization.WORD,
            index_filterable=True
        )
    ],
    inverted_index_config=Configure.inverted_index(
        bm25_b=0.7,
        bm25_k1=1.25,
        index_null_state=True,
        index_property_length=True,
        index_timestamps=True,
        stopwords_preset=StopwordsPreset.EN,
        stopwords_additions=["1","2","3","4","5","6","7","8","9","0"],
    ),
    )

    boericke_collection = client.collections.use("Boericke_materia_medica_docs")
    with boericke_collection.batch.fixed_size(batch_size=200) as batch:
        for obj in data_objects:
            batch.add_object(properties=obj['properties'], vector=obj['vector'], uuid=generate_uuid5(obj['properties']))
            if batch.number_errors > 10:
                print("Batch import stopped due to excessive errors.")
                break 
    failed_objects = boericke_collection.batch.failed_objects
    if failed_objects:
        print(f"Number of failed imports: {len(failed_objects)}")
        print(f"First failed object: {failed_objects[0]}")
    print(f"Imported & vectorized {len(boericke_collection)} objects into the Boericke_materia_medica_docs collection")

c:\Users\hp\OneDrive\Desktop\rag_bot\.venv\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\hp\AppData\Local\Temp\ipykernel_8184\637124241.py:6: ResourceWarning: unclosed <socket.socket fd=4308, family=23, type=1, proto=0, laddr=('::1', 57850, 0, 0), raddr=('::1', 8080, 0, 0)>
  with weaviate.connect_to_local() as client:


Imported & vectorized 1030 objects into the Boericke_materia_medica_docs collection


In [ ]:
import weaviate
client = weaviate.connect_to_local()

In [ ]:
client.collections.delete("Reference_1_docs")

In [108]:
from weaviate.classes.query import BM25Operator, MetadataQuery, GroupBy
Boericke_materia_medica_collection = client.collections.use("Boericke_materia_medica_docs")
# group_by = GroupBy(
#     prop="Heading",
#     objects_per_group=3,
#     number_of_groups=2,
# )
response = Boericke_materia_medica_collection.query.bm25(
    query="What do you know about Southernwood",
    operator=BM25Operator.or_(minimum_match=1),
    query_properties=["Heading"],
    return_metadata=MetadataQuery(score=True),
    auto_limit=1
    #group_by=group_by
)
for o in response.objects:
    print(o.properties)
    print(o.metadata.score)

{'content': 'Skin. --Eruptions come out on face; are suppressed, and the skin becomes purplish. Skin flabby and loose. Furuncles. Falling out of hair. Itching chilblains. Modalities. --Worse, cold air, checked secretions. Better, motion.\nRelationship. --Compare: Scrophularia; Bryonia; Stellaria; Benzoic acid, in gout. Iodine, Natr mur in marasmus.\nDose. --Third to thirtieth potency.', 'filename': 'Boericke_materia_medica.pdf', 'heading': 'Southernwood', 'page_No': 8}
3.485743284225464
{'content': 'A very useful remedy in marasmus, especially of lower extremities only, yet with good appetite. Metastasis. Rheumatism following checked diarrhœa. Ill effects of suppressed conditions especially in gouty subjects. Tuberculous peritonitis. Exudative pleurisy and other exudative processes. After operation upon the chest for hydrothorax or empyæmia, a pressing sensation remains. Aggravation\nof hæmorrhoids when rheumatism improves. Nosebleed and hydrocele in boys. Great weakness after influenz

In [29]:
Boericke_materia_medica_collection = client.collections.use("Boericke_materia_medica_docs")
vector=model.encode("What are the characteristic abdominal symptoms of Colocynthis, and what kind of patient is it most suitable for?").tolist()
norm = math.sqrt(sum(x*x for x in vector))
normalized_vector = [x / norm for x in vector]
response = Boericke_materia_medica_collection.query.near_vector(
    near_vector=normalized_vector,
    auto_limit=1,
    return_metadata=MetadataQuery(distance=True)
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.distance)

{'content': 'Marked symptoms in intestinal tract. Pain in nape of neck. Paralytic weakness. Small warts on hands.\nStomach.––During a meal, flatulence; afterwards lassitude taciturn and hot, pain in epigastrium, especially on breathing.\nAbdomen. – –Movements and grumbling in abdomen. Loose evacuations with much flatulency, especially left side with pullings in legs. Abundant and frequent emission of fetid flatus.\nDose. – –Third potency.', 'filename': 'Boericke_materia_medica.pdf', 'heading': 'Loadstone', 'page_No': 244}
-0.5438779592514038


In [107]:
from weaviate.classes.query import HybridFusion
from weaviate.classes.query import BM25Operator, MetadataQuery, GroupBy

query = """What do you know about Southernwood"""
Boericke_materia_medica_collection = client.collections.use("Boericke_materia_medica_docs")
vector=model.encode(query).tolist()
norm = math.sqrt(sum(x*x for x in vector))
normalized_vector = [x / norm for x in vector]

response = Boericke_materia_medica_collection.query.hybrid(
    query=query, 
    auto_limit=1,
    bm25_operator=BM25Operator.or_(minimum_match=1),
    query_properties=["Heading"],
    fusion_type=HybridFusion.RELATIVE_SCORE,
    alpha=1,  # optional, defaults to 0.5 (equal weighting)
    vector=normalized_vector
    )

for o in response.objects:
    print(o.properties)

{'content': 'Antidotes: Bathing with milk and Grindelia lotion very effective. Ampelopsis Trifolia-Three-leaf Woodbine--(Toxic dermatitis due to vegetable poisons-30 and 200. Very similar to Rhus poisoning). Desensitizing against Ivy poisoning by the use of descending doses of the tincture by mouth or by hypodermic injections is recommended by old school authorities, but is not as effective as the homeopathic remedies especially Rhus 30 and 200 and Anacard, etc. Anacard; Croton; Grindelia; Mezer; Cyprip; Plumbago (eczema of vulva); Graph. Compare: Rhus radicans (almost identical action); characteristics are, burning in tongue, tip feels sore, pains are often semilateral and in various parts, often remote and successive. Many symptoms are better after a storm has thoroughly set in, especially after an electric storm. Has pronounced yearly aggravation (Laches). Rhus radicans has headache in occiput even pain in nape of neck and from there pains draw over the head forwards. Rhus diversilo

In [30]:
from weaviate.classes.query import HybridFusion
from weaviate.classes.query import BM25Operator, MetadataQuery, GroupBy

Boericke_materia_medica_collection = client.collections.use("Boericke_materia_medica_docs")
vector=model.encode("What are the characteristic abdominal symptoms of Colocynthis, and what kind of patient is it most suitable for?").tolist()
norm = math.sqrt(sum(x*x for x in vector))
normalized_vector = [x / norm for x in vector]

response = Boericke_materia_medica_collection.query.hybrid(
    query="What are the characteristic abdominal symptoms of Colocynthis, and what kind of patient is it most suitable for?", 
    auto_limit=1,
    bm25_operator=BM25Operator.or_(minimum_match=1),
    query_properties=["Heading"],
    fusion_type=HybridFusion.RELATIVE_SCORE,
    alpha=0.7,  # optional, defaults to 0.5 (equal weighting)
    vector=normalized_vector
    )

for o in response.objects:
    print(o.properties)

{'content': 'Marked symptoms in intestinal tract. Pain in nape of neck. Paralytic weakness. Small warts on hands.\nStomach.––During a meal, flatulence; afterwards lassitude taciturn and hot, pain in epigastrium, especially on breathing.\nAbdomen. – –Movements and grumbling in abdomen. Loose evacuations with much flatulency, especially left side with pullings in legs. Abundant and frequent emission of fetid flatus.\nDose. – –Third potency.', 'filename': 'Boericke_materia_medica.pdf', 'heading': 'Loadstone', 'page_No': 244}


In [ ]:
from weaviate.classes.query import HybridFusion
from weaviate.classes.query import BM25Operator, MetadataQuery, GroupBy


query="""Both CouchDB and MongoDB are document stores, but they differ in one infrastructure behavior. What is it?"""
Reference_1_collection = client.collections.use("Reference_1_docs")
vector=model.encode(query).tolist()
norm = math.sqrt(sum(x*x for x in vector))
normalized_vector = [x / norm for x in vector]

response = Reference_1_collection.query.hybrid(
    query=query, 
    auto_limit=1,
    limit=50,
    bm25_operator=BM25Operator.or_(minimum_match=2),
    query_properties=["Heading"],
    fusion_type=HybridFusion.RELATIVE_SCORE,
    alpha=0.7,  # optional, defaults to 0.5 (equal weighting)
    vector=normalized_vector
    )

for o in response.objects:
    print(o.properties)

c:\Users\hp\OneDrive\Desktop\rag_bot\.venv\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\hp\AppData\Local\Temp\ipykernel_8184\18851392.py:6: ResourceWarning: unclosed <socket.socket fd=4772, family=23, type=1, proto=0, laddr=('::1', 56133, 0, 0), raddr=('::1', 8080, 0, 0)>
  Reference_1_collection = client.collections.use("Reference_1_docs")


{'content': '(a)  Apache CouchDB and MongoDB. Apache CouchDB [91] and MongoDB [90] are two examples of document stores. Both of them provide a schema-less store where the primary objects are documents, organized into a collection of key-value fi  elds. The value of each fi  eld can be of type-string, integer, fl  oat, date, or an array of values. The databases expose a RESTful interface and represents data in JSON format. Both of them allow querying and indexing data by using the MapReduce programming model, expose Javascript as a base language for data querying and manipulation rather than SQL, and support large fi  les as documents. From an infrastructure point of view, the two systems support data replication and high-availability. CouchDB ensures ACID properties on data. MongoDB supports sharding , which is the ability to distribute the content of a collection among different nodes.\n(b)  Amazon Dynamo. Dynamo [92] is the distributed key-value store supporting the management of inf